In [ ]:
import pandas as pd
from urllib.parse import urlparse

# Đường dẫn file
input_file_path = r"c:\csv\kynghequynhgia.csv"
clean_output_path = r"C:\csv\kynghequynhgia-clean.csv"
bad_output_path = r"C:\csv\kynghequynhgia-bad.csv"
sql_output_path = r"C:\csv\kynghequynhgia-bad.sql"

# Danh sách *từ khóa* để loại bỏ (loại domain lành hợp lệ)
exclude_keywords = [
    "google",
    "youtube",
    "tiktok",
    "facebook",
    "instagram",
    "linkedin",
    "zalo",
    "gachhoavancnc",
    "dsdhome",
    "ecoever",
    "hatari",
]

# 1. Đọc CSV
df = pd.read_csv(input_file_path, header=None, names=["count","url","status"], dtype=str)

# 2. Lọc URL https://
df = df[df["url"].str.startswith("https://", na=False)]

# 3. Extract domain chính (scheme + host)
df["domain"] = df["url"].apply(lambda x: urlparse(x).scheme + "://" + urlparse(x).netloc)

# 4. Loại trùng và sort A → Z
df_unique = df.drop_duplicates(subset="domain").sort_values(by="domain")

# 5. Tách host (hostname) từ domain
df_unique["host"] = df_unique["domain"].apply(lambda x: urlparse(x).netloc.lower())

# 6. Lọc *domain sạch* theo từ khóa
mask_clean = df_unique["host"].apply(
    lambda host: any(keyword in host for keyword in exclude_keywords)
)

# domain_clean: chứa domain chứa các từ khóa *loại bỏ*
df_clean = df_unique[mask_clean]

# domain_bad: tất cả domain còn lại
df_bad = df_unique[~mask_clean]

# 7. Xuất CSV
df_clean[["domain"]].to_csv(clean_output_path, index=False)
df_bad[["domain"]].to_csv(bad_output_path, index=False)

# 8. Tạo SQL insert cho *domain bẩn*
with open(sql_output_path, "w", encoding="utf-8") as f:
    f.write("-- SQL Insert for Bad Domains\n")
    f.write("INSERT INTO bad_domains (domain) VALUES\n")
    sql_values = [f"('{d}')" for d in df_bad["domain"]]
    f.write(",\n".join(sql_values) + ";")

print("  🌐 Clean domains:", clean_output_path)
print("  🚫 Bad domains:", bad_output_path)
print("  💾 SQL file:", sql_output_path)


  🌐 Clean domains: C:\csv\kynghequynhgia-clean.csv
  🚫 Bad domains: C:\csv\kynghequynhgia-bad.csv
  💾 SQL file: C:\csv\kynghequynhgia-bad.sql


In [23]:
import pandas as pd
from pathlib import Path

# =========================
# CONFIG
# =========================
bad_domains_path = r"C:\csv\kynghequynhgia-bad.csv"
db_name = "gachhoavan_1"
sql_output_path = r"C:\csv\delete_bad_records.sql"

DOMAIN_COL = "domain"
BATCH_SIZE = 500

# =========================
# HELPERS
# =========================
def normalize_host(s: str) -> str:
    s = str(s).strip()
    if not s or s.lower() in {"nan", "none"}:
        return ""
    s = s.lower()

    if s.startswith("http://"):
        s = s[7:]
    elif s.startswith("https://"):
        s = s[8:]

    while s.startswith("//"):
        s = s[2:]

    for sep in ["/", "?", "#"]:
        if sep in s:
            s = s.split(sep, 1)[0]

    s = s.strip().strip(".")
    if s.startswith("www."):
        s = s[4:]
    return s

def sql_escape(s: str) -> str:
    return s.replace("\\", "\\\\").replace("'", "''")

def write_line(f, text=""):
    f.write(text + "\n")

# =========================
# LOAD + CLEAN DOMAINS
# =========================
df_bad = pd.read_csv(bad_domains_path)
if DOMAIN_COL not in df_bad.columns:
    raise ValueError(f"CSV missing column '{DOMAIN_COL}'. Found: {list(df_bad.columns)}")

hosts = sorted(set(filter(None, (normalize_host(x) for x in df_bad[DOMAIN_COL].tolist()))))
if not hosts:
    raise ValueError("No valid hosts found after normalization.")

Path(sql_output_path).parent.mkdir(parents=True, exist_ok=True)

with open(sql_output_path, "w", encoding="utf-8") as f:
    write_line(f, f"USE `{db_name}`;")
    write_line(f, "SET NAMES utf8mb4;")
    write_line(f, "SET CHARACTER SET utf8mb4;")
    write_line(f)

    # 1) Temp table chứa domain bẩn
    write_line(f, "DROP TEMPORARY TABLE IF EXISTS tmp_bad_domains;")
    write_line(f, "CREATE TEMPORARY TABLE tmp_bad_domains (")
    write_line(f, "  host VARCHAR(255) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci PRIMARY KEY")
    write_line(f, ") ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;")
    write_line(f)

    for i in range(0, len(hosts), BATCH_SIZE):
        batch = hosts[i:i + BATCH_SIZE]
        values = ",\n".join([f"('{sql_escape(h)}')" for h in batch])
        write_line(f, "INSERT IGNORE INTO tmp_bad_domains(host) VALUES")
        write_line(f, values + ";")
        write_line(f)

    # 2) Bảng gom record cần xóa (dựa trên PK)
    #    table_name + pk_col + pk_value sẽ giúp delete chuẩn
    write_line(f, "DROP TABLE IF EXISTS bad_link_rows;")
    write_line(f, "CREATE TABLE bad_link_rows (")
    write_line(f, "  id BIGINT UNSIGNED AUTO_INCREMENT PRIMARY KEY,")
    write_line(f, "  table_name VARCHAR(64) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,")
    write_line(f, "  pk_col VARCHAR(64) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,")
    write_line(f, "  pk_value VARCHAR(255) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,")
    write_line(f, "  matched_host VARCHAR(255) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL,")
    write_line(f, "  matched_col VARCHAR(64) CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci NOT NULL")
    write_line(f, ") ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;")
    write_line(f)

    # 3) Procedure: collect tất cả record có domain bẩn vào bad_link_rows
    write_line(f, "DROP PROCEDURE IF EXISTS sp_collect_bad_rows;")
    write_line(f, "DELIMITER $$")
    write_line(f)
    write_line(f, "CREATE PROCEDURE sp_collect_bad_rows()")
    write_line(f, "BEGIN")
    write_line(f, "  DECLARE done INT DEFAULT 0;")
    write_line(f, "  DECLARE v_table VARCHAR(64);")
    write_line(f, "  DECLARE v_col   VARCHAR(64);")
    write_line(f, "  DECLARE v_pkcol VARCHAR(64);")
    write_line(f)
    write_line(f, "  DECLARE cur CURSOR FOR")
    write_line(f, "    SELECT c.TABLE_NAME, c.COLUMN_NAME")
    write_line(f, "    FROM INFORMATION_SCHEMA.COLUMNS c")
    write_line(f, f"    WHERE c.TABLE_SCHEMA = '{sql_escape(db_name)}'")
    write_line(f, "      AND c.DATA_TYPE IN ('varchar','text','char','mediumtext','longtext','tinytext');")
    write_line(f)
    write_line(f, "  DECLARE CONTINUE HANDLER FOR NOT FOUND SET done = 1;")
    write_line(f)
    write_line(f, "  OPEN cur;")
    write_line(f)
    write_line(f, "  read_loop: LOOP")
    write_line(f, "    FETCH cur INTO v_table, v_col;")
    write_line(f, "    IF done = 1 THEN LEAVE read_loop; END IF;")
    write_line(f)
    write_line(f, "    SET v_pkcol = NULL;")
    write_line(f, "    SELECT k.COLUMN_NAME")
    write_line(f, "      INTO v_pkcol")
    write_line(f, "    FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE k")
    write_line(f, f"    WHERE k.TABLE_SCHEMA = '{sql_escape(db_name)}'")
    write_line(f, "      AND k.TABLE_NAME = v_table")
    write_line(f, "      AND k.CONSTRAINT_NAME = 'PRIMARY'")
    write_line(f, "    ORDER BY k.ORDINAL_POSITION")
    write_line(f, "    LIMIT 1;")
    write_line(f)
    write_line(f, "    -- Chỉ collect các table có PRIMARY KEY (delete chuẩn)")
    write_line(f, "    IF v_pkcol IS NOT NULL THEN")
    write_line(f, "      SET @sql = CONCAT(")
    write_line(f, "        'INSERT IGNORE INTO bad_link_rows(table_name, pk_col, pk_value, matched_host, matched_col) ',")
    write_line(f, "        'SELECT ',")
    write_line(f, "          QUOTE(v_table), ', ', QUOTE(v_pkcol), ', ',")
    write_line(f, "          'CAST(t.`', v_pkcol, '` AS CHAR(255)) AS pk_value, ',")
    write_line(f, "          'bd.host AS matched_host, ', QUOTE(v_col), ' AS matched_col ',")
    write_line(f, "        'FROM `', " + f"'{sql_escape(db_name)}'" + ", '`.`', v_table, '` t ',")
    write_line(f, "        'JOIN tmp_bad_domains bd ON ',")
    write_line(f, "          'CONVERT(t.`', v_col, '` USING utf8mb4) COLLATE utf8mb4_unicode_ci ',")
    write_line(f, "          'LIKE CONCAT(''%'', bd.host COLLATE utf8mb4_unicode_ci, ''%'') ',")
    write_line(f, "        'WHERE t.`', v_col, '` IS NOT NULL' ")
    write_line(f, "      );")
    write_line(f, "      PREPARE stmt FROM @sql;")
    write_line(f, "      EXECUTE stmt;")
    write_line(f, "      DEALLOCATE PREPARE stmt;")
    write_line(f, "    END IF;")
    write_line(f)
    write_line(f, "  END LOOP;")
    write_line(f, "  CLOSE cur;")
    write_line(f, "END$$")
    write_line(f)
    write_line(f, "DELIMITER ;")
    write_line(f)

    # 4) Procedure: delete theo bảng bad_link_rows (theo từng table)
    write_line(f, "DROP PROCEDURE IF EXISTS sp_delete_bad_rows;")
    write_line(f, "DELIMITER $$")
    write_line(f)
    write_line(f, "CREATE PROCEDURE sp_delete_bad_rows()")
    write_line(f, "BEGIN")
    write_line(f, "  DECLARE done INT DEFAULT 0;")
    write_line(f, "  DECLARE v_table VARCHAR(64);")
    write_line(f, "  DECLARE v_pkcol VARCHAR(64);")
    write_line(f)
    write_line(f, "  DECLARE cur2 CURSOR FOR")
    write_line(f, "    SELECT DISTINCT table_name, pk_col")
    write_line(f, "    FROM bad_link_rows;")
    write_line(f)
    write_line(f, "  DECLARE CONTINUE HANDLER FOR NOT FOUND SET done = 1;")
    write_line(f)
    write_line(f, "  OPEN cur2;")
    write_line(f, "  del_loop: LOOP")
    write_line(f, "    FETCH cur2 INTO v_table, v_pkcol;")
    write_line(f, "    IF done = 1 THEN LEAVE del_loop; END IF;")
    write_line(f)
    write_line(f, "    SET @dsql = CONCAT(")
    write_line(f, "      'DELETE t FROM `', " + f"'{sql_escape(db_name)}'" + ", '`.`', v_table, '` t ',")
    write_line(f, "      'JOIN bad_link_rows r ',")
    write_line(f, "        'ON r.table_name = ', QUOTE(v_table), ' ',")
    write_line(f, "       'AND r.pk_col = ', QUOTE(v_pkcol), ' ',")
    write_line(f, "       'AND CAST(t.`', v_pkcol, '` AS CHAR(255)) = r.pk_value' ")
    write_line(f, "    );")
    write_line(f)
    write_line(f, "    PREPARE dstmt FROM @dsql;")
    write_line(f, "    EXECUTE dstmt;")
    write_line(f, "    DEALLOCATE PREPARE dstmt;")
    write_line(f, "  END LOOP;")
    write_line(f, "  CLOSE cur2;")
    write_line(f, "END$$")
    write_line(f)
    write_line(f, "DELIMITER ;")
    write_line(f)

    # 5) Run: collect -> review -> delete
    write_line(f, "-- 1) Collect record dính link bẩn (theo PK)")
    write_line(f, "TRUNCATE TABLE bad_link_rows;")
    write_line(f, "CALL sp_collect_bad_rows();")
    write_line(f)
    write_line(f, "-- 2) Review nhanh (đếm theo table)")
    write_line(f, "SELECT table_name, COUNT(*) AS hits FROM bad_link_rows GROUP BY table_name ORDER BY hits DESC;")
    write_line(f)
    write_line(f, "-- 3) Nếu OK thì chạy DELETE")
    write_line(f, "-- CALL sp_delete_bad_rows();")
    write_line(f)
    write_line(f, "-- 4) Kiểm tra còn gì không")
    write_line(f, "-- SELECT table_name, COUNT(*) AS hits FROM bad_link_rows GROUP BY table_name ORDER BY hits DESC;")

print("Generated SQL file:")
print(sql_output_path)
print("Unique hosts:", len(hosts))
print("Note: Open the SQL file, run it in phpMyAdmin. First review, then uncomment CALL sp_delete_bad_rows().")

Generated SQL file:
C:\csv\delete_bad_records.sql
Unique hosts: 458
Note: Open the SQL file, run it in phpMyAdmin. First review, then uncomment CALL sp_delete_bad_rows().
